<a href="https://colab.research.google.com/github/eceirem/COVID19-Pneumonia-XRay-Classification/blob/main/notebooks/03_dl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
03_dl_pipeline_final.py
Author: Ece
Description: High-Performance DL Pipeline.
- Auto-checks Drive for existing models (skips training if exists).
- Unique layer naming to prevent naming conflicts.
"""

import os
import gc
import zipfile
import numpy as np
import tensorflow as tf

# --- GPU SETUP ---
if len(tf.config.list_physical_devices('GPU')) > 0:
    print("[INFO] 🔥 GPU ACTIVATED! Enabling Mixed Precision.")
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

from tensorflow.keras.applications import ResNet50, DenseNet201, EfficientNetV2B0, Xception, ConvNeXtTiny
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# --- CONFIGURATIONS ---
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 20
CLASSES = ["COVID", "Normal", "Viral Pneumonia"]
NUM_CLASSES = len(CLASSES)

DRIVE_MASKED_ZIP = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Dataset/Preprocessed/Maskeli/Full-Data_Preprocessed_Dataset.zip"
DRIVE_NOT_MASKED_ZIP = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Dataset/Preprocessed/Maskesiz/Full-Data_Preprocessed_Dataset_NotMasked.zip"
MODEL_SAVE_DIR = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Models"

def extract_datasets():
    os.makedirs("/content/local_datasets/Masked", exist_ok=True)
    os.makedirs("/content/local_datasets/NotMasked", exist_ok=True)
    if os.path.exists(DRIVE_MASKED_ZIP):
        with zipfile.ZipFile(DRIVE_MASKED_ZIP, 'r') as z: z.extractall("/content/local_datasets/Masked")
    if os.path.exists(DRIVE_NOT_MASKED_ZIP):
        with zipfile.ZipFile(DRIVE_NOT_MASKED_ZIP, 'r') as z: z.extractall("/content/local_datasets/NotMasked")

class ModelFactory:
    @staticmethod
    def _get_base(model_func, name):
        tf.keras.backend.clear_session()
        base = model_func(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
        base.trainable = True
        for layer in base.layers[:-30]: layer.trainable = False
        # Manuel unik isimler
        for i, layer in enumerate(base.layers): layer._name = f"{name}_{i}"
        return base

    @staticmethod
    def build_model(model_func, name):
        base = ModelFactory._get_base(model_func, name)
        x = GlobalAveragePooling2D()(base.output)
        x = Dropout(0.5)(x)
        out = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
        return Model(inputs=base.input, outputs=out, name=name)

    @staticmethod
    def build_hybrid():
        tf.keras.backend.clear_session()
        inp = Input(shape=(*IMG_SIZE, 3))

        r = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
        r.trainable = False
        for i, l in enumerate(r.layers): l._name = f"resnet_hybrid_{i}"

        d = DenseNet201(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
        d.trainable = False
        for i, l in enumerate(d.layers): l._name = f"densenet_hybrid_{i}"

        f1 = GlobalAveragePooling2D()(r(inp))
        f2 = GlobalAveragePooling2D()(d(inp))

        merged = Concatenate()([f1, f2])
        x = Dense(256, activation='relu')(merged)
        out = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
        return Model(inputs=inp, outputs=out, name="Hybrid_ResNet_DenseNet")

def run_training_pipeline(dataset_name, dataset_path):
    # Dataset root'u bul
    for root, dirs, _ in os.walk(dataset_path):
        if all(c in dirs for c in CLASSES):
            dataset_path = root
            break

    datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    train_gen = datagen.flow_from_directory(dataset_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, subset='training')
    val_gen = datagen.flow_from_directory(dataset_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, subset='validation')

    weights = compute_class_weight('balanced', classes=np.unique(train_gen.classes), y=train_gen.classes)
    cw = dict(enumerate(weights))

    models = {
        "ResNet50": lambda: ModelFactory.build_model(ResNet50, "ResNet50"),
        "DenseNet201": lambda: ModelFactory.build_model(DenseNet201, "DenseNet201"),
        "EfficientNetV2": lambda: ModelFactory.build_model(EfficientNetV2B0, "EfficientNetV2"),
        "Xception": lambda: ModelFactory.build_model(Xception, "Xception"),
        "ConvNeXt_Tiny": lambda: ModelFactory.build_model(ConvNeXtTiny, "ConvNeXt_Tiny"),
        "Hybrid": ModelFactory.build_hybrid
    }

    for name, func in models.items():
        save_path = os.path.join(MODEL_SAVE_DIR, f"{dataset_name}_{name}.keras")

        # IF-ELSE KONTROLÜ
        if os.path.exists(save_path):
            print(f"[SKIP] Model {name} exists, skipping...")
            continue

        print(f"--- Training {name} ---")
        model = func()
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

        model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, class_weight=cw,
                  callbacks=[ModelCheckpoint(save_path, monitor='val_auc', save_best_only=True, mode='max'),
                             EarlyStopping(monitor='val_auc', patience=4, restore_best_weights=True)])
        del model
        tf.keras.backend.clear_session()
        gc.collect()

if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive')
    extract_datasets()
    run_training_pipeline("MASKED", "/content/local_datasets/Masked")
    run_training_pipeline("NOT_MASKED", "/content/local_datasets/NotMasked")

In [ ]:
"""
04_evaluation.py
Author: Ece
Description: Automated evaluation pipeline for trained models.
Uses 'subset=validation' to evaluate models strictly on the hold-out set
used during training (consistent with training seed/split).
"""

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- CONFIGURATION ---
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
CLASSES = ["COVID", "Normal", "Viral Pneumonia"]
MODELS_DIR = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Models"
RESULTS_PATH = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Results/DL_Ablation_Results.xlsx"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

# Dataset paths used during training
MASKED_VAL_PATH = "/content/local_datasets/Masked"
NOT_MASKED_VAL_PATH = "/content/local_datasets/NotMasked"

def get_validation_generator(dataset_path):
    """
    Creates a generator that replicates the validation split used during training.
    validation_split=0.2 and subset='validation' ensures we test on unseen data.
    """
    # Finding the root dir (where COVID/Normal/Viral folders reside)
    root_dir = dataset_path
    for root, dirs, _ in os.walk(dataset_path):
        if all(c in dirs for c in CLASSES):
            root_dir = root
            break

    datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    return datagen.flow_from_directory(
        root_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation', # Matches the training hold-out set
        shuffle=False
    )

def evaluate_all_models():
    all_results = []
    model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.keras')]

    for model_file in model_files:
        model_path = os.path.join(MODELS_DIR, model_file)

        # Decide dataset based on model filename prefix
        data_path = MASKED_VAL_PATH if "MASKED_" in model_file else NOT_MASKED_VAL_PATH

        print(f"\n[INFO] Evaluating: {model_file}...")
        val_gen = get_validation_generator(data_path)
        model = tf.keras.models.load_model(model_path)

        # Predict
        preds = model.predict(val_gen)
        y_pred = np.argmax(preds, axis=1)
        y_true = val_gen.classes

        # Metrics
        report = classification_report(y_true, y_pred, output_dict=True, target_names=CLASSES)

        all_results.append({
            "Model": model_file,
            "Accuracy": report['accuracy'],
            "F1-Macro": report['macro avg']['f1-score'],
            "Precision-Macro": report['macro avg']['precision'],
            "Recall-Macro": report['macro avg']['recall']
        })

        # CM Plot
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
        plt.title(f"CM: {model_file}")
        plt.show()

        del model
        tf.keras.backend.clear_session()

    # Save Excel
    df = pd.DataFrame(all_results)
    df.to_excel(RESULTS_PATH, index=False)
    print(f"\n[SUCCESS] Final evaluation results saved to: {RESULTS_PATH}")
    return df

if __name__ == "__main__":
    final_df = evaluate_all_models()
    print(final_df)